### Fine tuning mistral 7b model

In [1]:
!pip install --upgrade sagemaker datasets --quiet

In [30]:
model_id, model_version = "huggingface-llm-huggingfaceh4-mistral-7b-sft-alpha", "1.2.0"

### Deploying the model - pretrained

In [ ]:
from sagemaker.jumpstart.model import JumpStartModel

pretrained_model = JumpStartModel(model_id=model_id, model_version=model_version)
pretrained_predictor = pretrained_model.deploy(instance_type="ml.g5.xlarge", accept_eula=True)

Using model 'huggingface-llm-huggingfaceh4-mistral-7b-sft-alpha' with version '1.2.0'. You can upgrade to version '1.2.2' to get the latest model specifications. Note that models may have different input/output signatures after a major version upgrade.
INFO:sagemaker.jumpstart:Using model 'huggingface-llm-huggingfaceh4-mistral-7b-sft-alpha' with version '1.2.0'. You can upgrade to version '1.2.2' to get the latest model specifications. Note that models may have different input/output signatures after a major version upgrade.
No instance type selected for inference hosting endpoint. Defaulting to ml.g5.2xlarge.
INFO:sagemaker.jumpstart:No instance type selected for inference hosting endpoint. Defaulting to ml.g5.2xlarge.
INFO:sagemaker:Creating model with name: huggingfaceh4-mistral-7b-sft-alpha-2024-11-17-19-47-44-696
INFO:sagemaker:Creating endpoint-config with name huggingfaceh4-mistral-7b-sft-alpha-2024-11-17-19-47-44-701
INFO:sagemaker:Creating endpoint with name huggingfaceh4-mist

--

In [7]:
import sagemaker
# execution role for the endpoint
role = sagemaker.get_execution_role()

# sagemaker session for interacting with different AWS APIs
sess = sagemaker.session.Session()

# Region
region_name = sess._region_name

print(f"sagemaker role arn: {role}")
print(f"sagemaker session region: {region_name}")

sagemaker role arn: arn:aws:iam::054037115303:role/service-role/AmazonSageMaker-ExecutionRole-20241003T204056
sagemaker session region: us-west-2


In [8]:
from sagemaker.predictor import Predictor

endpoint_name = pretrained_predictor.endpoint_name

llm = Predictor(
    endpoint_name='huggingfaceh4-mistral-7b-sft-alpha-2024-11-17-19-47-44-701',
    sagemaker_session=sess,
    serializer=sagemaker.serializers.JSONSerializer(),
    deserializer=sagemaker.deserializers.JSONDeserializer(),
)

In [19]:
def print_response(payload, response):
    print(payload["inputs"])
    print(f"> {response[0]}")
    print("\n==================================\n")

In [22]:
payload = {
    "inputs": "what is finance?",
    "temperature": 0.6,
    "top_p": 0.9,
    "max_tokens": 512,
}
try:
    response = llm.predict(
        payload
    )
    print_response(payload, response)
except Exception as e:
    print(e)

what is finance?
> {'generated_text': 'what is finance? The word “finance” comes from the French and is composed of two words, “fin” meaning “end” or “end” and the word “pre-mum” which means “beginning".\n\nAnother interpretation is the latter, this could be taken literally in the sense that finance means money, but rather in a less literal sense. this takes on the aspect of transiency in the act of creating, meaning that money at best can be envied rather than earned.\n\nThe root of finance'}




In [ ]:
from datasets import load_dataset

finbro_dataset = load_dataset("'/kaggle/input/engineering-colleges-in-india/engineering colleges in India.csv'", split="train")

# For demonstration purposes of this tutorial, we train our model with 5% of the whole dataset. The test data is used to evaluate at the end.
train_and_test_dataset = finbro_dataset.train_test_split(test_size=0.95)  # train_size=0.9, test_size=0.1

# Dumping the training data to a local file to be used for training.
train_and_test_dataset["train"].to_json("train.jsonl")

Creating json from Arrow format:   0%|          | 0/20 [00:00<?, ?ba/s]

20887168

In [ ]:
train_and_test_dataset['train']

In [24]:
train_and_test_dataset['train'][0]

{'input': 'You are a financial expert. Your task is to provide accurate and insightful answers to finance-related questions based on the given conversation context.',
 'instruction': "User:UnitedHealth Group Incorporated (UNH) investment in AI and machine learning R&D.\nAssistant:Regarding UnitedHealth Group Incorporated's investment in AI and machine learning R&D the company has been actively exploring these technologies to enhance their healthcare services. Although exact figures on their investments are not readily available UnitedHealth Group has dedicated significant resources to research and development in the field of AI and machine learning.\n\nUser:Could you provide any specific projects or initiatives undertaken by UnitedHealth Group in the realm of AI and machine learning?\nAssistant:UnitedHealth Group has initiated several projects and initiatives in AI and machine learning. One notable example is their collaboration with Optum Labs a research and innovation center. Togethe

In [25]:
import json

template = {
    "prompt": "### Input:\n{input}\n\n### Instruction:\n{instruction}\n\n",
    "completion": "{output}",
}
with open("template.json", "w") as f:
    json.dump(template, f)

### saving it to S3

In [26]:
from sagemaker.s3 import S3Uploader
import sagemaker
import random

output_bucket = sagemaker.Session().default_bucket()
local_data_file = "train.jsonl"
train_data_location = f"s3://{output_bucket}/finbro_dataset"
S3Uploader.upload(local_data_file, train_data_location)
S3Uploader.upload("template.json", train_data_location)
print(f"Training data: {train_data_location}")

Training data: s3://sagemaker-us-west-2-054037115303/finbro_dataset


In [27]:
from sagemaker.jumpstart.estimator import JumpStartEstimator


estimator = JumpStartEstimator(
    model_id=model_id,
    model_version=model_version,
    environment={"accept_eula": "true"},
    disable_output_compression=True,
    instance_type="ml.g5.2xlarge",  # For Llama-3.2 3b, add instance_type = "ml.g5.12xlarge"
)
# By default, instruction tuning is set to false. Thus, to use instruction tuning dataset you use
estimator.set_hyperparameters(
    instruction_tuned="True", epoch="5", max_input_length="1024"
)
estimator.fit({"training": train_data_location})

INFO:sagemaker:Creating training-job with name: hf-llm-gemma-2b-2024-11-17-18-02-28-310
ERROR:sagemaker:Please check the troubleshooting guide for common errors: https://docs.aws.amazon.com/sagemaker/latest/dg/sagemaker-python-sdk-troubleshooting.html#sagemaker-python-sdk-troubleshooting-create-training-job


ResourceLimitExceeded: An error occurred (ResourceLimitExceeded) when calling the CreateTrainingJob operation: The account-level service limit 'ml.g5.2xlarge for training job usage' is 0 Instances, with current utilization of 0 Instances and a request delta of 1 Instances. Please use AWS Service Quotas to request an increase for this quota. If AWS Service Quotas is not available, contact AWS support to request an increase for this quota.

### Deploying to new endpoint

In [ ]:
finetuned_predictor = estimator.deploy(instance_type="ml.g5.xlarge")

### validating the result

In [ ]:
import pandas as pd
from IPython.display import display, HTML

test_dataset = train_and_test_dataset["test"]

(
    inputs,
    ground_truth_responses,
    responses_before_finetuning,
    responses_after_finetuning,
) = (
    [],
    [],
    [],
    [],
)


def predict_and_print(datapoint):
    # For instruction fine-tuning, we insert a special key between input and output
    input_output_demarkation_key = "\n\n### Response:\n"

    payload = {
        "inputs": template["prompt"].format(
            instruction=datapoint["instruction"], input=datapoint["input"]
        )
        + input_output_demarkation_key,
        "parameters": {"max_new_tokens": 100},
    }
    inputs.append(payload["inputs"])
    ground_truth_responses.append(datapoint["output"])
    # Please change the following line to "accept_eula=true"
    pretrained_response = pretrained_predictor.predict(
        payload, custom_attributes="accept_eula=false"
    )
    responses_before_finetuning.append(pretrained_response.get("generated_text"))
    # Fine Tuned Llama 3.2 models doesn't required to set "accept_eula=true"
    finetuned_response = finetuned_predictor.predict(payload)
    responses_after_finetuning.append(finetuned_response.get("generated_text"))


try:
    for i, datapoint in enumerate(test_dataset.select(range(5))):
        predict_and_print(datapoint)

    df = pd.DataFrame(
        {
            "Inputs": inputs,
            "Ground Truth": ground_truth_responses,
            "Response from non-finetuned model": responses_before_finetuning,
            "Response from fine-tuned model": responses_after_finetuning,
        }
    )
    display(HTML(df.to_html()))
except Exception as e:
    print(e)

In [ ]:
# Delete resources
pretrained_predictor.delete_model()
pretrained_predictor.delete_endpoint()
finetuned_predictor.delete_model()
finetuned_predictor.delete_endpoint()